In [1]:
import os

# KerasHub dùng TensorFlow
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import pandas as pd
from pathlib import Path

import tensorflow as tf
import keras_hub

from sklearn.model_selection import train_test_split

from tensorflow.keras import Sequential, Input, Model
from tensorflow.keras.layers import (
    Flatten,
    Dense,
    Dropout,
    GlobalAveragePooling2D,
    BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.vgg16 import (
    preprocess_input as vgg_preprocess
)

tf.keras.utils.set_random_seed(845)

In [2]:
data_dir = Path(
    "/kaggle/input/datasets/ashfakyeafi/"
    "cat-dog-images-for-classification"
)

image_dir = data_dir / "cat_dog"

df = pd.read_csv(data_dir / "cat_dog.csv")

X = df["image"]
y = df["labels"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=845,
    stratify=y
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=2/3,
    random_state=845,
    stratify=y_temp
)
print("Thư mục ảnh tồn tại:", image_dir.is_dir())

Thư mục ảnh tồn tại: True


In [3]:
resnet_preset = "hf://keras/resnet_18_imagenet"

resnet_converter = keras_hub.layers.ResNetImageConverter.from_preset(
    resnet_preset,
    image_size=(224, 224)
)


def preprocess_image(image, model_name):
    image = tf.cast(image, tf.float32)

    if model_name == "mlp":
        image = tf.image.resize(image, (64, 64))
        return image / 255.0

    elif model_name == "vgg":
        image = tf.image.resize(image, (224, 224))
        return vgg_preprocess(image)

    elif model_name == "resnet":
        return resnet_converter(image)

    else:
        raise ValueError(f"Not found: {model_name}")

config.json:   0%|          | 0.00/836 [00:00<?, ?B/s]

image_converter.json:   0%|          | 0.00/873 [00:00<?, ?B/s]

In [4]:
def make_dataset(filenames, labels, model_name, training=False):
    # Ghép thư mục với tên file thành đường dẫn đầy đủ
    paths = [str(image_dir / name) for name in filenames]

    # Chuyển nhãn thành số thực để dùng với BCE
    labels = np.asarray(labels, dtype=np.float32)

    # Ghép từng đường dẫn với nhãn tương ứng
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    # Chỉ xáo thứ tự tập train
    if training:
        dataset = dataset.shuffle(
            buffer_size=len(paths),
            seed=845,
            reshuffle_each_iteration=True
        )

    def load_example(path, label):
        # Đọc file thành dữ liệu ảnh RGB
        image = tf.io.read_file(path)

        image = tf.io.decode_image(
            image,
            channels=3,
            expand_animations=False
        )

        image.set_shape((None, None, 3))

        # Xử lý đúng theo model
        image = preprocess_image(image, model_name)

        # Một ảnh có một nhãn
        label = tf.reshape(label, (1,))

        return image, label

    # Áp dụng hàm đọc cho từng ảnh
    dataset = dataset.map(
        load_example,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # Gom ảnh thành batch và chuẩn bị trước các batch tiếp theo
    dataset = dataset.batch(32)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

In [5]:
datasets = {}

for name in ["mlp", "vgg", "resnet"]:
    datasets[name] = {
        "train": make_dataset(
            X_train, y_train, name, training=True
        ),
        "valid": make_dataset(
            X_valid, y_valid, name
        ),
        "test": make_dataset(
            X_test, y_test, name
        )
    }

I0000 00:00:1793205759.410872      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793205759.415056      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [6]:
for name in ["mlp", "vgg", "resnet"]:
    images, labels = next(iter(datasets[name]["valid"]))

    print(f"\nModel: {name}")
    print("Image shape:", images.shape)
    print("Label shape:", labels.shape)
    print("5 nhãn đầu:", labels[:5].numpy().ravel())


Model: mlp
Image shape: (32, 64, 64, 3)
Label shape: (32, 1)
5 nhãn đầu: [0. 0. 0. 1. 1.]

Model: vgg
Image shape: (32, 224, 224, 3)
Label shape: (32, 1)
5 nhãn đầu: [0. 0. 0. 1. 1.]

Model: resnet
Image shape: (32, 224, 224, 3)
Label shape: (32, 1)
5 nhãn đầu: [0. 0. 0. 1. 1.]


In [7]:
# #MLP
mlp_model = Sequential([
    Input(shape=(64, 64, 3)),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

mlp_model.summary()

mlp_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

mlp_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

mlp_history = mlp_model.fit(
    datasets["mlp"]["train"],
    validation_data=datasets["mlp"]["valid"],
    epochs=20,
    callbacks=[mlp_early_stopping],
    verbose=2
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 12288)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,572,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,581,313 (6.03 MB)

 Trainable params: 1,581,313 (6.03 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20


I0000 00:00:1793205777.322949      69 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


547/547 - 2548s - 5s/step - accuracy: 0.5385 - loss: 0.6913 - precision: 0.5384 - recall: 0.5394 - val_accuracy: 0.5912 - val_loss: 0.6714 - val_precision: 0.5707 - val_recall: 0.7360
Epoch 2/20
547/547 - 11s - 21ms/step - accuracy: 0.5743 - loss: 0.6762 - precision: 0.5684 - recall: 0.6171 - val_accuracy: 0.5720 - val_loss: 0.6745 - val_precision: 0.5429 - val_recall: 0.9120
Epoch 3/20
547/547 - 11s - 21ms/step - accuracy: 0.5857 - loss: 0.6716 - precision: 0.5713 - recall: 0.6864 - val_accuracy: 0.6148 - val_loss: 0.6618 - val_precision: 0.5877 - val_recall: 0.7696
Epoch 4/20
547/547 - 11s - 21ms/step - accuracy: 0.5974 - loss: 0.6656 - precision: 0.5837 - recall: 0.6798 - val_accuracy: 0.6312 - val_loss: 0.6548 - val_precision: 0.6128 - val_recall: 0.7128
Epoch 5/20
547/547 - 11s - 21ms/step - accuracy: 0.6018 - loss: 0.6636 - precision: 0.5857 - recall: 0.6960 - val_accuracy: 0.6064 - val_loss: 0.6488 - val_precision: 0.5755 - val_recall: 0.8112
Epoch 6/20
547/547 - 12s - 21ms/step

In [8]:
mlp_results = mlp_model.evaluate(
    datasets["mlp"]["valid"],
    return_dict=True,
    verbose=0
)

for name, value in mlp_results.items():
    print(f"{name}: {value:.4f}")

accuracy: 0.6396
loss: 0.6419
precision: 0.6333
recall: 0.6632


In [9]:
# #VGG-16
vgg_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

vgg_base.trainable = False

vgg_model = Sequential([
    Input(shape=(224, 224, 3)),
    vgg_base,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

vgg_model.summary()

vgg_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

vgg_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

vgg_head_history = vgg_model.fit(
    datasets["vgg"]["train"],
    validation_data=datasets["vgg"]["valid"],
    epochs=20,
    callbacks=[vgg_early_stopping],
    verbose=2
)

vgg_results = vgg_model.evaluate(
    datasets["vgg"]["valid"],
    return_dict=True,
    verbose=0
)

print("VGG16 — Validation")
print(f"Loss:      {vgg_results['loss']:.4f}")
print(f"Accuracy:  {vgg_results['accuracy']:.2%}")
print(f"Precision: {vgg_results['precision']:.2%}")
print(f"Recall:    {vgg_results['recall']:.2%}")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,780,481 (56.38 MB)

 Trainable params: 65,793 (257.00 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/20
547/547 - 147s - 269ms/step - accuracy: 0.9691 - loss: 0.1097 - precision: 0.9696 - recall: 0.9687 - val_accuracy: 0.9828 - val_loss: 0.0506 - val_precision: 0.9756 - val_recall: 0.9904
Epoch 2/20
547/547 - 114s - 208ms/step - accuracy: 0.9841 - loss: 0.0462 - precision: 0.9852 - recall: 0.9829 - val_accuracy: 0.9816 - val_loss: 0.0618 - val_precision: 0.9725 - val_recall: 0.9912
Epoch 3/20
547/547 - 113s - 207ms/step - accuracy: 0.9865 - loss: 0.0357 - precision: 0.9870 - recall: 0.9861 - val_accuracy: 0.9828 - val_loss: 0.0531 - val_precision: 0.9771 - val_recall: 0.9888
Epoch 4/20
547/547 - 113s - 207ms/step - accuracy: 0.9894 - loss: 0.0292 - precision: 0.9894 - recall: 0.9895 - val_accuracy: 0.9844 - val_loss: 0.0519 - val_precision: 0.9879 - val_recall: 0.9808
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.
VGG16 — Validation
Loss:      0.0506
Accuracy:  98.28%
Precision: 97.56%
Recall:    99.04%


In [10]:
#Resnet18
resnet_base = keras_hub.models.ResNetBackbone.from_preset(
    resnet_preset
)

resnet_base.trainable = False

resnet_model = Sequential([
    Input(shape=(224, 224, 3)),
    resnet_base,
    GlobalAveragePooling2D(),
    Dense(1, activation="sigmoid")
])

resnet_model.summary()

resnet_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

resnet_head_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

resnet_head_history = resnet_model.fit(
    datasets["resnet"]["train"],
    validation_data=datasets["resnet"]["valid"],
    epochs=20,
    callbacks=[resnet_head_stopping],
    verbose=2
)

resnet_head_results = resnet_model.evaluate(
    datasets["resnet"]["valid"],
    return_dict=True,
    verbose=0
)

model.weights.h5: reconstructing file:   0%|          |  0.00B / 44.9MB            

model.weights.h5: downloading bytes:           |  0.00B            

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ res_net_backbone                │ (None, 7, 7, 512)      │    11,186,112 │
│ (ResNetBackbone)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,186,625 (42.67 MB)

 Trainable params: 513 (2.00 KB)

 Non-trainable params: 11,186,112 (42.67 MB)

Epoch 1/20
547/547 - 38s - 69ms/step - accuracy: 0.8013 - loss: 0.5297 - precision: 0.8336 - recall: 0.7529 - val_accuracy: 0.9520 - val_loss: 0.3628 - val_precision: 0.9242 - val_recall: 0.9848
Epoch 2/20
547/547 - 27s - 50ms/step - accuracy: 0.9594 - loss: 0.2823 - precision: 0.9304 - recall: 0.9930 - val_accuracy: 0.9680 - val_loss: 0.2198 - val_precision: 0.9459 - val_recall: 0.9928
Epoch 3/20
547/547 - 27s - 50ms/step - accuracy: 0.9718 - loss: 0.1861 - precision: 0.9503 - recall: 0.9957 - val_accuracy: 0.9736 - val_loss: 0.1555 - val_precision: 0.9554 - val_recall: 0.9936
Epoch 4/20
547/547 - 27s - 50ms/step - accuracy: 0.9766 - loss: 0.1381 - precision: 0.9594 - recall: 0.9954 - val_accuracy: 0.9772 - val_loss: 0.1202 - val_precision: 0.9635 - val_recall: 0.9920
Epoch 5/20
547/547 - 27s - 50ms/step - accuracy: 0.9796 - loss: 0.1098 - precision: 0.9655 - recall: 0.9947 - val_accuracy: 0.9796 - val_loss: 0.0981 - val_precision: 0.9687 - val_recall: 0.9912
Epoch 6/20
547/547 - 28s 

In [11]:
resnet_results = resnet_model.evaluate(
    datasets["resnet"]["valid"],
    return_dict=True,
    verbose=0
)

print("ResNet18 — Validation")
print(f"Loss:      {resnet_results['loss']:.4f}")
print(f"Accuracy:  {resnet_results['accuracy']:.2%}")
print(f"Precision: {resnet_results['precision']:.2%}")
print(f"Recall:    {resnet_results['recall']:.2%}")

ResNet18 — Validation
Loss:      0.0352
Accuracy:  98.92%
Precision: 99.04%
Recall:    98.80%


In [12]:
%pip install -q gradio
import gradio as gr

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [13]:
def predict_image(image):
    if image is None:
        return None, None, None

    # Chuyển ảnh được upload thành RGB
    image = np.asarray(image.convert("RGB"))

    models = {
        "mlp": mlp_model,
        "vgg": vgg_model,
        "resnet": resnet_model
    }

    results = []

    for name, model in models.items():
        # Dùng đúng tiền xử lý của từng model
        processed_image = preprocess_image(image, name)

        # Thêm chiều batch: một ảnh → batch gồm một ảnh
        batch = tf.expand_dims(processed_image, axis=0)

        # Dự đoán, không train
        prediction = model(batch, training=False)
        dog_probability = float(prediction.numpy().reshape(-1)[0])

        results.append({
            "Mèo": 1 - dog_probability,
            "Chó": dog_probability
        })

    return tuple(results)

In [14]:
demo = gr.Interface(
    fn=predict_image,

    inputs=gr.Image(
        type="pil",
        label="Tải ảnh chó hoặc mèo"
    ),

    outputs=[
        gr.Label(label="MLP", num_top_classes=2),
        gr.Label(label="VGG16", num_top_classes=2),
        gr.Label(label="ResNet18", num_top_classes=2)
    ],

    title="Phân loại chó và mèo",

    description=(
        "Tải một ảnh để so sánh dự đoán của ba model. "
        "Phần trăm hiển thị là xác suất dự đoán, không phải accuracy."
    )
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://1a43f19863398ae48f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
